# A/B 测试统计分析

本笔记本对 A/B 测试数据进行全面的统计分析。

分析内容包括：
- 转化率对比
- 卡方检验
- Odds Ratio 与置信区间
- 效应量（Cohen's h）
- 比例差值及其置信区间
- 事后功效分析
- 样本量估算
- 时间趋势分析

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, contingency, norm
from datetime import datetime

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../data/ab_data_cleaned.csv')
df.head()

## 1. 核心指标

In [ ]:
group_result = df.groupby('group').agg(
    users=('user_id', 'count'),
    conversions=('converted', 'sum'),
)
group_result['conversion_rate'] = group_result['conversions'] / group_result['users']

control_cvr = group_result.loc['control', 'conversion_rate']
treat_cvr = group_result.loc['treatment', 'conversion_rate']

abs_uplift = treat_cvr - control_cvr
rel_uplift = (treat_cvr / control_cvr) - 1

print('===== 核心指标 =====')
print(group_result)
print(f'\n旧版转化率: {control_cvr:.4f}')
print(f'新版转化率: {treat_cvr:.4f}')
print(f'绝对提升: {abs_uplift:.4f}')
print(f'相对提升: {rel_uplift:.4f} ({rel_uplift*100:.2f}%)')

## 2. 统计显著性检验

In [ ]:
control_conv = group_result.loc['control', 'conversions']
control_noconv = group_result.loc['control', 'users'] - control_conv
treat_conv = group_result.loc['treatment', 'conversions']
treat_noconv = group_result.loc['treatment', 'users'] - treat_conv

table = [
    [int(control_conv), int(control_noconv)],
    [int(treat_conv), int(treat_noconv)]
]

chi2, p, dof, expected = chi2_contingency(table)

odds_result = contingency.odds_ratio(table)
odds_ratio = odds_result.statistic
ci_low_or, ci_high_or = odds_result.confidence_interval(confidence_level=0.95)

print('===== 卡方检验 =====')
print(f'chi2 = {chi2:.4f}')
print(f'p = {p:.4f}')
print(f'alpha = 0.05 水平下: {"显著" if p < 0.05 else "不显著"}')
print(f'\nOdds Ratio = {odds_ratio:.4f}')
print(f'95% CI for OR: [{ci_low_or:.4f}, {ci_high_or:.4f}]')

## 3. 效应量 Cohen's h

公式: $h = 2 \arcsin(\sqrt{p_2}) - 2 \arcsin(\sqrt{p_1})$

判断标准: $|h| < 0.2$ 可忽略, $0.2 \sim 0.5$ 小效应, $> 0.5$ 中到大效应

In [ ]:
h = 2 * np.arcsin(np.sqrt(treat_cvr)) - 2 * np.arcsin(np.sqrt(control_cvr))

if abs(h) < 0.2:
    interpretation = '可忽略'
elif abs(h) < 0.5:
    interpretation = '小效应'
else:
    interpretation = '中到大效应'

print(f"Cohen's h = {h:.6f}")
print(f'效应量判断: {interpretation}')
print(f'\n判断标准: < 0.2 可忽略, 0.2~0.5 小效应, > 0.5 中到大效应')

## 4. 比例差值的置信区间

公式: $SE = \sqrt{\frac{p_1(1-p_1)}{n_1} + \frac{p_2(1-p_2)}{n_2}}$

$95\% CI = (\hat{p}_2 - \hat{p}_1) \pm z_{\alpha/2} \cdot SE$

In [ ]:
n_ctrl = group_result.loc['control', 'users']
n_treat = group_result.loc['treatment', 'users']

alpha = 0.05
z_alpha = norm.ppf(1 - alpha/2)

diff = treat_cvr - control_cvr
se_diff = np.sqrt(control_cvr*(1-control_cvr)/n_ctrl + treat_cvr*(1-treat_cvr)/n_treat)
ci_low_diff = diff - z_alpha * se_diff
ci_high_diff = diff + z_alpha * se_diff

print(f'比例差值 = {diff:.6f} ({diff*100:.2f}%)')
print(f'SE = {se_diff:.6f}')
print(f'95% CI: [{ci_low_diff:.6f}, {ci_high_diff:.6f}]')
print(f'95% CI (%): [{ci_low_diff*100:.2f}%, {ci_high_diff*100:.2f}%]')
print(f'\n置信区间包含 0: {"是 -- 无显著差异" if ci_low_diff < 0 < ci_high_diff else "否 -- 差异显著"}')

## 5. 功效分析与样本量估算

功效公式: $\text{Power} = \Phi\left(\frac{|\delta|}{SE_{pooled}} - z_{\alpha/2}\right)$

所需样本量 (80% power): $n = \frac{(z_{\alpha/2} + z_{0.8})^2 \cdot (p_1(1-p_1) + p_2(1-p_2))}{(p_1 - p_2)^2}$

In [ ]:
p_pool = (control_conv + treat_conv) / (n_ctrl + n_treat)
se_pooled = np.sqrt(p_pool*(1-p_pool)*(1/n_ctrl + 1/n_treat))

z_beta = abs(diff) / se_pooled - z_alpha
power = norm.cdf(z_beta)

z_beta_80 = norm.ppf(0.80)
n_per_group = (z_alpha + z_beta_80)**2 * (control_cvr*(1-control_cvr) + treat_cvr*(1-treat_cvr)) / diff**2

coverage = n_ctrl / n_per_group * 100
multiplier = n_per_group / n_ctrl

print('===== 功效分析 =====')
print(f'事后统计功效 = {power:.4f} ({power*100:.2f}%)')
print(f'\n达到 80% 功效所需样本量 (每组): {int(np.ceil(n_per_group)):,}')
print(f'当前样本量 (每组): {n_ctrl:,}')
print(f'覆盖率: {coverage:.1f}%')
print(f'\n要达到 80% 功效，需要 {multiplier:.1f} 倍数据量')
print(f'按当前日均流量 (~{n_ctrl/23:,.0f} 用户/天)，实验需运行约 {n_per_group/(n_ctrl/23):.0f} 天')

## 6. 可视化：转化率对比与差值置信区间

In [ ]:
labels = ['对照组 (旧版)', '实验组 (新版)']
conv_rates = [control_cvr, treat_cvr]
colors = ['#1f77b4', '#ff6b6b']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
bars = ax1.bar(labels, conv_rates, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
ax1.set_title('转化率对比', fontsize=14, fontweight='bold')
ax1.set_ylabel('转化率', fontsize=12)
ax1.set_ylim(0.10, 0.13)

for bar, rate in zip(bars, conv_rates):
    ax1.text(bar.get_x()+bar.get_width()/2., rate + 0.0005,
             f'{rate:.4f}', ha='center', fontsize=12, fontweight='bold')

ax2 = axes[1]
ax2.errorbar([0], [diff], yerr=[[abs(ci_low_diff)], [ci_high_diff]],
             fmt='o', color='#2ca02c', capsize=10, markersize=12, elinewidth=2)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
ax2.set_xticks([0])
ax2.set_xticklabels(['实验组 - 对照组'])
ax2.set_ylabel('比例差值', fontsize=12)
ax2.set_title('差值与 95% 置信区间', fontsize=14, fontweight='bold')
ax2.text(0, diff + 0.0003, f'{diff:.4f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../output/conversion_rate_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('图表已保存至 output/ 文件夹')

## 7. 时间趋势分析

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date

daily = df.groupby(['date', 'group'])['converted'].agg(['sum', 'count']).reset_index()
daily['rate'] = daily['sum'] / daily['count']

print(f'实验持续天数: {daily["date"].nunique()}')
print(f'日期范围: {daily["date"].min()} 至 {daily["date"].max()}')

In [ ]:
control_daily = daily[daily['group'] == 'control'].set_index('date')['rate']
treatment_daily = daily[daily['group'] == 'treatment'].set_index('date')['rate']

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax1 = axes[0]
ax1.plot(control_daily.index, control_daily.values, 'o-', color='#1f77b4', label='对照组', linewidth=2, markersize=6)
ax1.plot(treatment_daily.index, treatment_daily.values, 's-', color='#ff6b6b', label='实验组', linewidth=2, markersize=6)
ax1.axhline(y=control_cvr, color='#1f77b4', linestyle='--', alpha=0.5)
ax1.axhline(y=treat_cvr, color='#ff6b6b', linestyle='--', alpha=0.5)
ax1.set_title('每日转化率变化趋势', fontsize=14, fontweight='bold')
ax1.set_ylabel('转化率')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

df['week'] = df['timestamp'].dt.isocalendar().week.astype(int)
weekly = df.groupby(['week', 'group'])['converted'].agg(['sum', 'count']).reset_index()
weekly['rate'] = weekly['sum'] / weekly['count']

ax2 = axes[1]
weeks = sorted(weekly['week'].unique())
ctrl_weekly = weekly[weekly['group'] == 'control'].set_index('week')['rate']
treat_weekly = weekly[weekly['group'] == 'treatment'].set_index('week')['rate']

x = np.arange(len(weeks))
width = 0.35
bars1 = ax2.bar(x - width/2, [ctrl_weekly[w] for w in weeks], width, label='对照组', color='#1f77b4', edgecolor='white')
bars2 = ax2.bar(x + width/2, [treat_weekly[w] for w in weeks], width, label='实验组', color='#ff6b6b', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels([f'第 {w} 周' for w in weeks])
ax2.set_title('每周转化率对比', fontsize=14, fontweight='bold')
ax2.set_ylabel('转化率')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.0005, f'{bar.get_height():.4f}', ha='center', fontsize=9)
for bar in bars2:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.0005, f'{bar.get_height():.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../output/time_trend_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('时间趋势图已保存至 output/ 文件夹')

## 8. 最终结论

In [ ]:
print('='*60)
print('       A/B 测试最终结论')
print('='*60)

if p < 0.05:
    print('统计结果：差异显著')
    if treat_cvr > control_cvr:
        print('业务结论：新版页面显著提升转化率')
        print(f'建议：上线新版本，预计提升转化 {rel_uplift*100:.2f}%')
    else:
        print('业务结论：新版页面显著差于旧版')
        print('建议：不推荐上线，需重新优化设计')
else:
    print('统计结果：无显著差异 (p = {:.4f})'.format(p))
    print(f"效应量 (Cohen's h): {h:.4f} -- {interpretation}")
    print(f'统计功效: {power*100:.1f}% (建议 >= 80%)')
    print()
    print('关键发现：实验功效严重不足')
    print(f'当前样本 ({n_ctrl:,}/组) 仅为所需样本 {int(np.ceil(n_per_group)):,}/组 的 {coverage:.1f}%')
    print()
    print('建议：')
    print('1. 不应上线新版 -- 无证据表明其表现更好')
    print('2. 如需检测当前效应，需延长实验至每组 ~{:,} 用户'.format(int(np.ceil(n_per_group))))
    print('3. 重新评估 MDE -- 如果 0.16% 的差异在商业上无意义，')
    print('   当前样本已足以得出「无实际差异」的结论')
    print('4. 考虑细分分析，寻找不同人群中的差异化响应')

print('='*60)